### Part A

In [ ]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1. Load and Prepare the Dataset
df = pd.read_csv('garments_worker_productivity.csv')

df['department'] = df['department'].str.strip()

df['MeetsTarget'] = (df['actual_productivity'] >= df['targeted_productivity']).astype(int)

# Define feature matrix X and the two target arrays
X = df.drop(columns=['date', 'actual_productivity', 'MeetsTarget'])
y_reg = df['actual_productivity']
y_clf = df['MeetsTarget']

# 2. Unified Train-Test Split 
X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = train_test_split(
    X, y_reg, y_clf, test_size=0.2, random_state=42
)

# 3. Build the Scikit-learn Preprocessing Pipeline
categorical_cols = ['quarter', 'department', 'day', 'team']
numeric_cols = ['targeted_productivity', 'smv', 'wip', 'over_time', 'incentive', 
                'idle_time', 'idle_men', 'no_of_style_change', 'no_of_workers']

# Numeric Pipeline: Median Imputation (for 'wip' missing values) + Scaling
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical Pipeline: Mode Imputation + One-Hot Encoding
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine pipelines into a ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

# Fit and transform the training data; only transform the test data to prevent data leakage
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)


# 4. Regression: Train, Time, and Evaluate Linear Regression
print("--- Linear Regression (Scikit-learn) ---")
reg_model = LinearRegression()

# Time Training
start_train_reg = time.time()
reg_model.fit(X_train_processed, y_reg_train)
train_time_reg = time.time() - start_train_reg

# Time Prediction
start_pred_reg = time.time()
y_reg_pred = reg_model.predict(X_test_processed)
pred_time_reg = time.time() - start_pred_reg

# Evaluate
mae = mean_absolute_error(y_reg_test, y_reg_pred)
rmse = np.sqrt(mean_squared_error(y_reg_test, y_reg_pred))
r2 = r2_score(y_reg_test, y_reg_pred)

print(f"Training Time  : {train_time_reg:.5f} seconds")
print(f"Prediction Time: {pred_time_reg:.5f} seconds")
print(f"MAE: {mae:.4f} | RMSE: {rmse:.4f} | R2: {r2:.4f}\n")


# 5. Classification: Train, Time, and Evaluate Logistic Regression
print("--- Logistic Regression (Scikit-learn) ---")
clf_model = LogisticRegression(max_iter=1000, random_state=42)

# Time Training
start_train_clf = time.time()
clf_model.fit(X_train_processed, y_clf_train)
train_time_clf = time.time() - start_train_clf

# Time Prediction
start_pred_clf = time.time()
y_clf_pred = clf_model.predict(X_test_processed)
pred_time_clf = time.time() - start_pred_clf

# Evaluate
acc = accuracy_score(y_clf_test, y_clf_pred)
prec = precision_score(y_clf_test, y_clf_pred)
rec = recall_score(y_clf_test, y_clf_pred)
f1 = f1_score(y_clf_test, y_clf_pred)

print(f"Training Time  : {train_time_clf:.5f} seconds")
print(f"Prediction Time: {pred_time_clf:.5f} seconds")
print(f"Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1-Score: {f1:.4f}")

--- Linear Regression (Scikit-learn) ---
Training Time  : 0.10347 seconds
Prediction Time: 0.00000 seconds
MAE: 0.1106 | RMSE: 0.1495 | R2: 0.1584

--- Logistic Regression (Scikit-learn) ---
Training Time  : 0.02008 seconds
Prediction Time: 0.00000 seconds
Accuracy: 0.7542 | Precision: 0.7892 | Recall: 0.9096 | F1-Score: 0.8451


### Part B

In [ ]:
import pandas as pd
import numpy as np
import time

# 1. Manual Preprocessing (Imputation, Scaling, Encoding)
def manual_preprocess(X_train, X_test, num_cols, cat_cols):
    X_tr = X_train.copy()
    X_te = X_test.copy()

    # A. Missing Value Imputation
    # Numeric: Median imputation
    medians = X_tr[num_cols].median()
    X_tr[num_cols] = X_tr[num_cols].fillna(medians)
    X_te[num_cols] = X_te[num_cols].fillna(medians)

    # Categorical: Mode imputation
    modes = X_tr[cat_cols].mode().iloc[0]
    X_tr[cat_cols] = X_tr[cat_cols].fillna(modes)
    X_te[cat_cols] = X_te[cat_cols].fillna(modes)

    # B. Feature Scaling (Standardization)
    means = X_tr[num_cols].mean()
    stds = X_tr[num_cols].std(ddof=0) 
    
    # Avoid division by zero for constant columns
    stds[stds == 0] = 1 
    
    X_tr[num_cols] = (X_tr[num_cols] - means) / stds
    X_te[num_cols] = (X_te[num_cols] - means) / stds

    # C. One-Hot Encoding
    X_tr_cat = pd.get_dummies(X_tr[cat_cols], drop_first=False, dtype=float)
    X_te_cat = pd.get_dummies(X_te[cat_cols], drop_first=False, dtype=float)

    # Align columns to ensure train and test have identical dummy features
    X_tr_cat, X_te_cat = X_tr_cat.align(X_te_cat, join='left', axis=1, fill_value=0.0)

    # Combine numeric and encoded categorical features
    X_tr_final = pd.concat([X_tr[num_cols], X_tr_cat], axis=1).to_numpy()
    X_te_final = pd.concat([X_te[num_cols], X_te_cat], axis=1).to_numpy()

    # D. Add Bias Term (Intercept) - column of 1s at index 0
    X_tr_final = np.c_[np.ones(X_tr_final.shape[0]), X_tr_final]
    X_te_final = np.c_[np.ones(X_te_final.shape[0]), X_te_final]

    return X_tr_final, X_te_final

# Apply manual preprocessing to the raw splits from Part A
# (Assuming numeric_cols and categorical_cols are defined as in Part A)
X_train_np, X_test_np = manual_preprocess(X_train, X_test, numeric_cols, categorical_cols)

# Convert targets to numpy arrays for vectorized operations
y_reg_train_np = y_reg_train.to_numpy()
y_reg_test_np = y_reg_test.to_numpy()
y_clf_train_np = y_clf_train.to_numpy()
y_clf_test_np = y_clf_test.to_numpy()


# 2. Manual Evaluation Metrics
def calc_regression_metrics(y_true, y_pred):
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred)**2))
    
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum((y_true - np.mean(y_true))**2)
    r2 = 1 - (ss_res / ss_tot)
    
    return mae, rmse, r2

def calc_classification_metrics(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    
    acc = (tp + tn) / len(y_true)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * (prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
    
    return acc, prec, rec, f1


# 3. Manual Linear Regression (Closed-Form Solution)
class ManualLinearRegression:
    def fit(self, X, y):
        # Normal Equation: Theta = (X^T * X)^-1 * X^T * y
        # We use np.linalg.pinv (pseudo-inverse) to handle potential multicollinearity
        self.theta = np.linalg.pinv(X.T @ X) @ X.T @ y
        
    def predict(self, X):
        return X @ self.theta

print("--- Linear Regression (From Scratch) ---")
manual_reg = ManualLinearRegression()

# Time Training
start_train = time.time()
manual_reg.fit(X_train_np, y_reg_train_np)
reg_train_time = time.time() - start_train

# Time Prediction
start_pred = time.time()
y_reg_pred_np = manual_reg.predict(X_test_np)
reg_pred_time = time.time() - start_pred

# Evaluate
mae, rmse, r2 = calc_regression_metrics(y_reg_test_np, y_reg_pred_np)
print(f"Training Time  : {reg_train_time:.5f} seconds")
print(f"Prediction Time: {reg_pred_time:.5f} seconds")
print(f"MAE: {mae:.4f} | RMSE: {rmse:.4f} | R2: {r2:.4f}\n")


# 4. Manual Logistic Regression (Gradient Descent)
class ManualLogisticRegression:
    def __init__(self, lr=0.1, epochs=1000):
        self.lr = lr
        self.epochs = epochs
        self.theta = None
        
    def sigmoid(self, z):
        # Clip values to prevent np.exp overflow
        z = np.clip(z, -250, 250)
        return 1 / (1 + np.exp(-z))
        
    def fit(self, X, y):
        m, n = X.shape
        self.theta = np.zeros(n)
        
        # Vectorized Gradient Descent
        for _ in range(self.epochs):
            predictions = self.sigmoid(X @ self.theta)
            gradient = (1 / m) * (X.T @ (predictions - y))
            self.theta -= self.lr * gradient
            
    def predict(self, X, threshold=0.5):
        probabilities = self.sigmoid(X @ self.theta)
        return (probabilities >= threshold).astype(int)

print("--- Logistic Regression (From Scratch) ---")
manual_clf = ManualLogisticRegression(lr=0.1, epochs=1000)

# Time Training
start_train = time.time()
manual_clf.fit(X_train_np, y_clf_train_np)
clf_train_time = time.time() - start_train

# Time Prediction
start_pred = time.time()
y_clf_pred_np = manual_clf.predict(X_test_np)
clf_pred_time = time.time() - start_pred

# Evaluate
acc, prec, rec, f1 = calc_classification_metrics(y_clf_test_np, y_clf_pred_np)
print(f"Training Time  : {clf_train_time:.5f} seconds")
print(f"Prediction Time: {clf_pred_time:.5f} seconds")
print(f"Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1-Score: {f1:.4f}")

--- Linear Regression (From Scratch) ---
Training Time  : 0.00206 seconds
Prediction Time: 0.00000 seconds
MAE: 0.1088 | RMSE: 0.1481 | R2: 0.1736

--- Logistic Regression (From Scratch) ---
Training Time  : 0.06845 seconds
Prediction Time: 0.00000 seconds
Accuracy: 0.7625 | Precision: 0.7703 | Recall: 0.9661 | F1-Score: 0.8571


### Part C

In [ ]:
import pandas as pd
import numpy as np
import time
from IPython.display import display

# 1. Comparison Tables (Scikit-learn vs Manual)
# Recalculating metrics to capture both Scikit-learn and Manual models 
# (since Part B overwrote the metric variables from Part A)

# Regression
sk_mae, sk_rmse, sk_r2 = calc_regression_metrics(y_reg_test_np, y_reg_pred)
man_mae, man_rmse, man_r2 = calc_regression_metrics(y_reg_test_np, y_reg_pred_np)

reg_comparison = pd.DataFrame({
    'Implementation': ['Scikit-learn', 'Manual (Normal Equation)'],
    'MAE': [sk_mae, man_mae],
    'RMSE': [sk_rmse, man_rmse],
    'R2 Score': [sk_r2, man_r2],
    'Train Time (s)': [train_time_reg, reg_train_time],
    'Pred Time (s)': [pred_time_reg, reg_pred_time]
})

# Classification 
sk_acc, sk_prec, sk_rec, sk_f1 = calc_classification_metrics(y_clf_test_np, y_clf_pred)
man_acc, man_prec, man_rec, man_f1 = calc_classification_metrics(y_clf_test_np, y_clf_pred_np)

clf_comparison = pd.DataFrame({
    'Implementation': ['Scikit-learn', 'Manual (Gradient Descent)'],
    'Accuracy': [sk_acc, man_acc],
    'Precision': [sk_prec, man_prec],
    'Recall': [sk_rec, man_rec],
    'F1-Score': [sk_f1, man_f1],
    'Train Time (s)': [train_time_clf, clf_train_time],
    'Pred Time (s)': [pred_time_clf, clf_pred_time]
})

print("=== Regression Comparison ===")
display(reg_comparison)
print("\n=== Classification Comparison ===")
display(clf_comparison)

# 2. Optimization: Ridge Regression & Early-Stopping Logistic Regression
class OptimizedManualLinearRegression:
    def __init__(self, lambda_reg=0.1):
        self.lambda_reg = lambda_reg
        self.theta = None

    def fit(self, X, y):
        m, n = X.shape
        I = np.eye(n)
        I[0, 0] = 0  # Do not regularize the bias term
        
        # Closed-form Ridge Regression (L2 Penalty)
        self.theta = np.linalg.pinv(X.T @ X + self.lambda_reg * I) @ X.T @ y
        
    def predict(self, X):
        return X @ self.theta

class OptimizedManualLogisticRegression:
    def __init__(self, lr=0.5, epochs=3000, lambda_reg=0.1, tol=1e-5):
        self.lr = lr
        self.epochs = epochs
        self.lambda_reg = lambda_reg
        self.tol = tol
        self.theta = None
        
    def sigmoid(self, z):
        z = np.clip(z, -250, 250)
        return 1 / (1 + np.exp(-z))
        
    def fit(self, X, y):
        m, n = X.shape
        self.theta = np.zeros(n)
        prev_loss = float('inf')
        
        for i in range(self.epochs):
            predictions = self.sigmoid(X @ self.theta)
            
            # Gradient with L2 Regularization (excluding bias)
            reg_term = (self.lambda_reg / m) * self.theta
            reg_term[0] = 0 
            
            gradient = (1 / m) * (X.T @ (predictions - y)) + reg_term
            self.theta -= self.lr * gradient
            
            # Early stopping convergence check
            epsilon = 1e-15
            clipped_preds = np.clip(predictions, epsilon, 1 - epsilon)
            loss = -np.mean(y * np.log(clipped_preds) + (1 - y) * np.log(1 - clipped_preds))
            loss += (self.lambda_reg / (2 * m)) * np.sum(self.theta[1:]**2)
            
            if abs(prev_loss - loss) < self.tol:
                break
            prev_loss = loss
            
    def predict(self, X, threshold=0.5):
        probabilities = self.sigmoid(X @ self.theta)
        return (probabilities >= threshold).astype(int)

# 3. Train, Time, and Evaluate Optimized Models
print("\n--- Optimized Linear Regression (From Scratch + L2 Regularization) ---")
opt_reg = OptimizedManualLinearRegression(lambda_reg=1.0)
start_train = time.time()
opt_reg.fit(X_train_np, y_reg_train_np)
opt_reg_train_time = time.time() - start_train

start_pred = time.time()
y_opt_reg_pred = opt_reg.predict(X_test_np)
opt_reg_pred_time = time.time() - start_pred

opt_mae, opt_rmse, opt_r2 = calc_regression_metrics(y_reg_test_np, y_opt_reg_pred)
print(f"Training Time  : {opt_reg_train_time:.5f} seconds")
print(f"Prediction Time: {opt_reg_pred_time:.5f} seconds")
print(f"MAE: {opt_mae:.4f} | RMSE: {opt_rmse:.4f} | R2: {opt_r2:.4f}")

print("\n--- Optimized Logistic Regression (From Scratch + L2 + Early Stopping) ---")
opt_clf = OptimizedManualLogisticRegression(lr=0.5, epochs=3000, lambda_reg=0.1, tol=1e-5)
start_train = time.time()
opt_clf.fit(X_train_np, y_clf_train_np)
opt_clf_train_time = time.time() - start_train

start_pred = time.time()
y_opt_clf_pred = opt_clf.predict(X_test_np)
opt_clf_pred_time = time.time() - start_pred

opt_acc, opt_prec, opt_rec, opt_f1 = calc_classification_metrics(y_clf_test_np, y_opt_clf_pred)
print(f"Training Time  : {opt_clf_train_time:.5f} seconds")
print(f"Prediction Time: {opt_clf_pred_time:.5f} seconds")
print(f"Accuracy: {opt_acc:.4f} | Precision: {opt_prec:.4f} | Recall: {opt_rec:.4f} | F1-Score: {opt_f1:.4f}")

=== Regression Comparison ===


,Implementation,MAE,RMSE,R2 Score,Train Time (s),Pred Time (s)
0,Scikit-learn,0.110626,0.149485,0.158426,0.103474,0.0
1,Manual (Normal Equation),0.108771,0.148134,0.173576,0.002059,0.0



=== Classification Comparison ===


,Implementation,Accuracy,Precision,Recall,F1-Score,Train Time (s),Pred Time (s)
0,Scikit-learn,0.754167,0.789216,0.909605,0.845144,0.020082,0.0
1,Manual (Gradient Descent),0.762500,0.770270,0.966102,0.857143,0.068451,0.0



--- Optimized Linear Regression (From Scratch + L2 Regularization) ---
Training Time  : 0.00000 seconds
Prediction Time: 0.00000 seconds
MAE: 0.1086 | RMSE: 0.1480 | R2: 0.1749

--- Optimized Logistic Regression (From Scratch + L2 + Early Stopping) ---
Training Time  : 0.44279 seconds
Prediction Time: 0.00000 seconds
Accuracy: 0.7417 | Precision: 0.7406 | Recall: 1.0000 | F1-Score: 0.8510
